# Main Forecasting Models

## Objective
Develop and evaluate advanced forecasting models:
1. **SARIMA**: Statistical model with seasonal components
2. **SARIMAX**: SARIMA with exogenous variables (promotions)
3. **RandomForest**: Ensemble tree-based ML model
4. **GradientBoosting**: Advanced boosting algorithm

## Goals
- Outperform baseline models
- Capture trend, seasonality, and promotion effects
- Use proper time series validation (TimeSeriesSplit)
- Tune hyperparameters
- Interpret model behavior

In [1]:
# Import libraries
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import pickle

# Import project modules
import src.config as config
from src.utils import print_section_header
from src.models import (
    ARIMAForecaster,
    TreeBasedForecaster,
    print_stationarity_test
)

# Set random seed
np.random.seed(config.RANDOM_SEED)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')

print("Environment setup complete!")

Environment setup complete!


## 1. Load Data and Baseline Results

In [2]:
# Load preprocessed data
X_train = np.load(config.DATA_PATH / 'X_train.npy')
y_train = np.load(config.DATA_PATH / 'y_train.npy')
X_test = np.load(config.DATA_PATH / 'X_test.npy')
y_test = np.load(config.DATA_PATH / 'y_test.npy')
train_dates = np.load(config.DATA_PATH / 'train_dates.npy', allow_pickle=True)
test_dates = np.load(config.DATA_PATH / 'test_dates.npy', allow_pickle=True)

# Load feature names
with open(config.DATA_PATH / 'feature_names.txt', 'r') as f:
    feature_names = [line.strip() for line in f]

# Load baseline results for comparison
with open(config.DATA_PATH / 'baseline_results.pkl', 'rb') as f:
    baseline_results = pickle.load(f)

print(f"Data loaded successfully!")
print(f"Train: {X_train.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")
print(f"Features: {len(feature_names)}")

Data loaded successfully!
Train: 1404 samples
Test: 254 samples
Features: 21


In [3]:
# Define evaluation metrics (same as baseline)
def compute_metrics(y_true, y_pred):
    """Compute MAE and RMSE."""
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return {'MAE': mae, 'RMSE': rmse}

def print_metrics(model_name, metrics):
    """Print metrics in a formatted way."""
    print(f"\n{model_name}")
    print("-" * 50)
    print(f"  MAE:  {metrics['MAE']:.2f}")
    print(f"  RMSE: {metrics['RMSE']:.2f}")

## 2. SARIMA Model

### Stationarity Check

ARIMA models assume stationarity (constant mean and variance over time).  
We'll use the Augmented Dickey-Fuller (ADF) test to check.

In [4]:
# Check stationarity of training series
stationarity_result = print_stationarity_test(y_train, series_name="Training Sales")


Augmented Dickey-Fuller Test for Training Sales
Test Statistic: -5.0636
P-value: 0.0000
Number of lags used: 22
Number of observations: 1381

Critical Values:
  1%: -3.435
  5%: -2.864
  10%: -2.568

✓ Result: Series is STATIONARY (p-value < 0.05)


### ACF and PACF Plots

- **ACF (Autocorrelation Function)**: Helps determine MA order (q)
- **PACF (Partial Autocorrelation Function)**: Helps determine AR order (p)

We'll look for significant spikes to inform SARIMA parameters.

In [ ]:
# Plot ACF and PACF
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# ACF
plot_acf(y_train, lags=40, ax=axes[0])
axes[0].set_title('Autocorrelation Function (ACF)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Lag', fontsize=11)
axes[0].set_ylabel('Correlation', fontsize=11)

# PACF
plot_pacf(y_train, lags=40, ax=axes[1], method='ywm')
axes[1].set_title('Partial Autocorrelation Function (PACF)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Lag', fontsize=11)
axes[1].set_ylabel('Correlation', fontsize=11)

plt.tight_layout()

from src.utils import save_figure
save_figure(fig, '16_acf_pacf.png')
plt.show()

### Fit SARIMA Model

**SARIMA Parameters:**
- `order=(p, d, q)`: Non-seasonal parameters
  - p: AR order
  - d: Differencing order
  - q: MA order
- `seasonal_order=(P, D, Q, s)`: Seasonal parameters
  - P: Seasonal AR order
  - D: Seasonal differencing
  - Q: Seasonal MA order
  - s: Seasonal period (7 for weekly)

We'll start with SARIMA(1,1,1)×(1,1,1,7) based on ACF/PACF and weekly seasonality.

In [6]:
print_section_header("SARIMA Model Training")

# Initialize SARIMA model
sarima_model = ARIMAForecaster(
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7)
)

# Fit model
print("Fitting SARIMA model... (this may take a minute)")
sarima_model.fit(y_train)
print("✓ SARIMA model fitted successfully")

# Make predictions
sarima_pred = sarima_model.predict(steps=len(y_test))

# Evaluate
sarima_metrics = compute_metrics(y_test, sarima_pred)
print_metrics("SARIMA(1,1,1)×(1,1,1,7)", sarima_metrics)


 SARIMA Model Training

Fitting SARIMA model... (this may take a minute)
✓ SARIMA model fitted successfully

SARIMA(1,1,1)×(1,1,1,7)
--------------------------------------------------
  MAE:  2769.46
  RMSE: 3440.91


In [7]:
# Display model summary
print(sarima_model.get_model_summary())

                                     SARIMAX Results                                     
Dep. Variable:                                 y   No. Observations:                 1404
Model:             SARIMAX(1, 1, 1)x(1, 1, 1, 7)   Log Likelihood              -12864.215
Date:                           Sun, 25 Jan 2026   AIC                          25738.429
Time:                                   14:40:49   BIC                          25764.604
Sample:                                        0   HQIC                         25748.219
                                          - 1404                                         
Covariance Type:                             opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.2496      0.011     22.982      0.000       0.228       0.271
ma.L1         -1.0456      0.004   -281.434

## 3. SARIMAX with Exogenous Variables

Enhance SARIMA by including exogenous variables (promotions).  
This allows the model to account for external factors affecting sales.

In [8]:
# Extract promotion features for SARIMAX
# We'll use 'onpromotion' as the exogenous variable
promo_idx = feature_names.index('onpromotion')
X_train_promo = X_train[:, promo_idx].reshape(-1, 1)
X_test_promo = X_test[:, promo_idx].reshape(-1, 1)

print(f"Exogenous variable shape (train): {X_train_promo.shape}")
print(f"Exogenous variable shape (test): {X_test_promo.shape}")

Exogenous variable shape (train): (1404, 1)
Exogenous variable shape (test): (254, 1)


In [9]:
print_section_header("SARIMAX Model Training")

# Initialize SARIMAX model
sarimax_model = ARIMAForecaster(
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7)
)

# Fit model with exogenous variable
print("Fitting SARIMAX model with promotions... (this may take a minute)")
sarimax_model.fit(y_train, X_train=X_train_promo)
print("✓ SARIMAX model fitted successfully")

# Make predictions
sarimax_pred = sarimax_model.predict(steps=len(y_test), X_test=X_test_promo)

# Evaluate
sarimax_metrics = compute_metrics(y_test, sarimax_pred)
print_metrics("SARIMAX(1,1,1)×(1,1,1,7) + Promotions", sarimax_metrics)


 SARIMAX Model Training

Fitting SARIMAX model with promotions... (this may take a minute)
✓ SARIMAX model fitted successfully

SARIMAX(1,1,1)×(1,1,1,7) + Promotions
--------------------------------------------------
  MAE:  2278.69
  RMSE: 2987.62


## 4. RandomForest Model

**RandomForest**: Ensemble of decision trees
- Handles non-linear relationships
- Robust to outliers
- Provides feature importance
- Uses TimeSeriesSplit for proper validation

In [10]:
print_section_header("RandomForest Model Training")

# Initialize RandomForest
rf_params = {
    'n_estimators': 200,
    'max_depth': 20,
    'min_samples_split': 5,
    'min_samples_leaf': 2,
    'random_state': config.RANDOM_SEED,
    'n_jobs': -1
}

rf_model = TreeBasedForecaster(
    model_type='random_forest',
    params=rf_params,
    cv_splits=5,
    tune=False  # Set to True for hyperparameter tuning (slower)
)

# Fit model
print("Fitting RandomForest model...")
rf_model.fit(y_train, X_train)
print("✓ RandomForest model fitted successfully")

# Make predictions
rf_pred = rf_model.predict(X_test=X_test)

# Evaluate
rf_metrics = compute_metrics(y_test, rf_pred)
print_metrics("RandomForest", rf_metrics)


 RandomForest Model Training

Fitting RandomForest model...
✓ RandomForest model fitted successfully

RandomForest
--------------------------------------------------
  MAE:  1488.62
  RMSE: 2508.23


In [11]:
# Feature importance from RandomForest
rf_importance = rf_model.get_feature_importance(feature_names=feature_names)

print("\nTop 15 Most Important Features (RandomForest):")
print(rf_importance.head(15))


Top 15 Most Important Features (RandomForest):
            feature  importance
10      day_of_week    0.202188
15       is_weekend    0.182931
0       sales_lag_1    0.096316
8   rolling_mean_30    0.090743
4    rolling_mean_7    0.086360
2      sales_lag_14    0.055725
11     day_of_month    0.052758
1       sales_lag_7    0.051592
12            month    0.031079
6   rolling_mean_14    0.026739
5     rolling_std_7    0.026332
3      sales_lag_30    0.025952
18      onpromotion    0.018998
9    rolling_std_30    0.017514
7    rolling_std_14    0.015753


In [ ]:
# Visualize feature importance
fig, ax = plt.subplots(figsize=(10, 8))

rf_importance.head(15).plot(x='feature', y='importance', kind='barh', ax=ax,
                            color='steelblue', alpha=0.8, edgecolor='black', legend=False)
ax.set_xlabel('Importance', fontsize=12)
ax.set_ylabel('Feature', fontsize=12)
ax.set_title('RandomForest: Top 15 Feature Importances', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
ax.invert_yaxis()

plt.tight_layout()
save_figure(fig, '17_rf_feature_importance.png')
plt.show()

## 5. GradientBoosting Model

**GradientBoosting**: Sequential ensemble that builds trees to correct previous errors
- Often achieves better performance than RandomForest
- More sensitive to hyperparameters
- Can overfit if not tuned properly

In [13]:
print_section_header("GradientBoosting Model Training")

# Initialize GradientBoosting
gb_params = {
    'n_estimators': 200,
    'learning_rate': 0.1,
    'max_depth': 5,
    'min_samples_split': 5,
    'min_samples_leaf': 2,
    'random_state': config.RANDOM_SEED
}

gb_model = TreeBasedForecaster(
    model_type='gradient_boosting',
    params=gb_params,
    cv_splits=5,
    tune=False  # Set to True for hyperparameter tuning
)

# Fit model
print("Fitting GradientBoosting model...")
gb_model.fit(y_train, X_train)
print("✓ GradientBoosting model fitted successfully")

# Make predictions
gb_pred = gb_model.predict(X_test=X_test)

# Evaluate
gb_metrics = compute_metrics(y_test, gb_pred)
print_metrics("GradientBoosting", gb_metrics)


 GradientBoosting Model Training

Fitting GradientBoosting model...
✓ GradientBoosting model fitted successfully

GradientBoosting
--------------------------------------------------
  MAE:  1671.05
  RMSE: 2512.28


In [14]:
# Feature importance from GradientBoosting
gb_importance = gb_model.get_feature_importance(feature_names=feature_names)

print("\nTop 15 Most Important Features (GradientBoosting):")
print(gb_importance.head(15))


Top 15 Most Important Features (GradientBoosting):
            feature  importance
10      day_of_week    0.184250
15       is_weekend    0.145387
8   rolling_mean_30    0.092787
4    rolling_mean_7    0.081327
0       sales_lag_1    0.071329
2      sales_lag_14    0.069702
5     rolling_std_7    0.064779
1       sales_lag_7    0.057201
11     day_of_month    0.051393
12            month    0.033088
6   rolling_mean_14    0.031931
3      sales_lag_30    0.028144
7    rolling_std_14    0.026484
9    rolling_std_30    0.025836
18      onpromotion    0.021740


## 6. Model Comparison

In [15]:
# Combine all results
all_results = baseline_results['metrics'].copy()
all_results['SARIMA'] = sarima_metrics
all_results['SARIMAX'] = sarimax_metrics
all_results['RandomForest'] = rf_metrics
all_results['GradientBoosting'] = gb_metrics

# Create comparison table
results_df = pd.DataFrame(all_results).T
results_df = results_df.sort_values('MAE')

print_section_header("Complete Model Comparison")
print(results_df)

# Identify best model
best_model = results_df.index[0]
best_mae = results_df.loc[best_model, 'MAE']
best_rmse = results_df.loc[best_model, 'RMSE']

print(f"\n✓ Best performing model: {best_model}")
print(f"  MAE:  {best_mae:.2f}")
print(f"  RMSE: {best_rmse:.2f}")


 Complete Model Comparison

                                MAE         RMSE
RandomForest            1488.616848  2508.228777
Linear Regression       1669.273479  2556.153372
GradientBoosting        1671.046594  2512.275550
SARIMAX                 2278.688946  2987.621296
SARIMA                  2769.459164  3440.913312
Moving Average (7-day)  3416.858830  3924.052913
Seasonal Naive (7-day)  8466.059055  9175.241916
Naive                   8565.224409  9220.623933

✓ Best performing model: RandomForest
  MAE:  1488.62
  RMSE: 2508.23


In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(12, 8))

results_df.plot(kind='barh', ax=ax, alpha=0.8, edgecolor='black', width=0.7)
ax.set_xlabel('Error', fontsize=12)
ax.set_ylabel('Model', fontsize=12)
ax.set_title('Complete Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend(title='Metric', fontsize=11)
ax.grid(True, alpha=0.3, axis='x')
ax.invert_yaxis()

plt.tight_layout()
save_figure(fig, '18_complete_model_comparison.png')
plt.show()

## 7. Predictions Visualization

In [ ]:
# Plot all advanced model predictions
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

test_dates_dt = pd.to_datetime(test_dates)

# Plot 1: SARIMA
axes[0].plot(test_dates_dt, y_test, label='Actual', linewidth=2, alpha=0.8, color='black')
axes[0].plot(test_dates_dt, sarima_pred, label='Predicted', linewidth=1.5, alpha=0.8, color='purple')
axes[0].set_ylabel('Sales', fontsize=11)
axes[0].set_title(f'SARIMA (MAE: {sarima_metrics["MAE"]:.2f})', fontsize=12, fontweight='bold')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# Plot 2: SARIMAX
axes[1].plot(test_dates_dt, y_test, label='Actual', linewidth=2, alpha=0.8, color='black')
axes[1].plot(test_dates_dt, sarimax_pred, label='Predicted', linewidth=1.5, alpha=0.8, color='brown')
axes[1].set_ylabel('Sales', fontsize=11)
axes[1].set_title(f'SARIMAX (MAE: {sarimax_metrics["MAE"]:.2f})', fontsize=12, fontweight='bold')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

# Plot 3: RandomForest
axes[2].plot(test_dates_dt, y_test, label='Actual', linewidth=2, alpha=0.8, color='black')
axes[2].plot(test_dates_dt, rf_pred, label='Predicted', linewidth=1.5, alpha=0.8, color='green')
axes[2].set_ylabel('Sales', fontsize=11)
axes[2].set_title(f'RandomForest (MAE: {rf_metrics["MAE"]:.2f})', fontsize=12, fontweight='bold')
axes[2].legend(loc='upper right')
axes[2].grid(True, alpha=0.3)

# Plot 4: GradientBoosting
axes[3].plot(test_dates_dt, y_test, label='Actual', linewidth=2, alpha=0.8, color='black')
axes[3].plot(test_dates_dt, gb_pred, label='Predicted', linewidth=1.5, alpha=0.8, color='red')
axes[3].set_ylabel('Sales', fontsize=11)
axes[3].set_xlabel('Date', fontsize=11)
axes[3].set_title(f'GradientBoosting (MAE: {gb_metrics["MAE"]:.2f})', fontsize=12, fontweight='bold')
axes[3].legend(loc='upper right')
axes[3].grid(True, alpha=0.3)

plt.xticks(rotation=45)
plt.tight_layout()
save_figure(fig, '19_advanced_model_predictions.png')
plt.show()

## 8. Save Results for Final Evaluation

In [18]:
# Save all model results
main_model_results = {
    'predictions': {
        'sarima': sarima_pred,
        'sarimax': sarimax_pred,
        'random_forest': rf_pred,
        'gradient_boosting': gb_pred
    },
    'metrics': {
        'SARIMA': sarima_metrics,
        'SARIMAX': sarimax_metrics,
        'RandomForest': rf_metrics,
        'GradientBoosting': gb_metrics
    },
    'feature_importance': {
        'random_forest': rf_importance,
        'gradient_boosting': gb_importance
    },
    'best_model': {
        'name': best_model,
        'mae': best_mae,
        'rmse': best_rmse
    }
}

# Save to file
with open(config.DATA_PATH / 'main_model_results.pkl', 'wb') as f:
    pickle.dump(main_model_results, f)

# Also save all results combined
combined_results = {
    'all_metrics': all_results,
    'results_df': results_df,
    'best_model': best_model
}

with open(config.DATA_PATH / 'all_model_results.pkl', 'wb') as f:
    pickle.dump(combined_results, f)

print("\n✓ Model results saved successfully!")
print(f"  - main_model_results.pkl")
print(f"  - all_model_results.pkl")


✓ Model results saved successfully!
  - main_model_results.pkl
  - all_model_results.pkl


## 9. Summary

### Models Developed:

1. **SARIMA**: Statistical time series model with seasonality
2. **SARIMAX**: SARIMA enhanced with promotion information
3. **RandomForest**: Ensemble tree model with non-linear capabilities
4. **GradientBoosting**: Advanced boosting algorithm

### Key Findings:

- Best model: [Determined by results]
- Significant improvement over baselines: [Yes/No]
- Most important features: [From feature importance analysis]
- Promotion impact: [SARIMAX vs SARIMA comparison]

### Next Steps:

1. Comprehensive evaluation and error analysis
2. Model selection and interpretation
3. Business use case application (inventory planning)
4. Documentation and recommendations